# IgarapIA – treinar o modelo melhorado no Google Colab

Antes de começar: **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)**.

Este notebook treina o detector com **as 1.500 fotos do TACO**, o modelo **YOLO11s**, imagens de **640 px** e **100 épocas**. O primeiro modelo usou só 585 fotos, o YOLO11n, 512 px e 20 épocas em CPU. Tempo estimado: 15 a 20 min para baixar o TACO e cerca de 1 hora de treino.

In [ ]:
!git clone -q https://github.com/Igordev7/igarapia-UEA.git igarapia
%cd igarapia
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baixar o TACO completo (1.500 fotos)

In [ ]:
!git clone -q https://github.com/pedropro/TACO.git /content/TACO
!cd /content/TACO && python download.py

## 2. Preparar a base com as 5 classes do projeto

In [ ]:
!python scripts/preparar_taco.py --anotacoes /content/TACO/data/annotations.json --imagens /content/TACO/data --saida dados/igarapia --tam-max 1280

## 3. (Opcional, mas é o que mais melhora) Acrescentar fotos de lixo na água

O TACO tem pouco lixo boiando na água, e por isso o modelo confunde garrafas amassadas com latas. Para corrigir isso:

- **Fotos de igarapés anotadas pela equipe:** anotem de 100 a 200 fotos no Roboflow ou no CVAT, com as classes `garrafa_pet`, `sacola_filme_plastico`, `isopor`, `lata` e `outros`, e exportem em formato **YOLO**.
- **Bases públicas de lixo flutuante:** no Roboflow Universe, procurem por *floating waste* ou *river plastic* e exportem em formato **YOLOv8**.

Enviem o `.zip` exportado na célula abaixo. No `--mapa`, liguem os nomes das classes da base enviada às classes do IgarapIA. As classes que ficarem fora do mapa vão para `outros`.

In [ ]:
from google.colab import files
import zipfile, glob, os
enviado = files.upload()
for nome in enviado:
    zipfile.ZipFile(nome).extractall('bases/extra')
print(glob.glob('bases/extra/**/data.yaml', recursive=True))

In [ ]:
# ajuste o caminho do data.yaml e o mapa de classes conforme a base enviada
!python scripts/adicionar_base_yolo.py --origem bases/extra/data.yaml --destino dados/igarapia \
    --mapa "bottle=garrafa_pet,plastic bottle=garrafa_pet,can=lata,plastic bag=sacola_filme_plastico,styrofoam=isopor" \
    --prefixo extra
!rm -f dados/igarapia/labels/*.cache

## 4. Treinar

In [ ]:
!python scripts/treinar.py --dados dados/igarapia/data.yaml --modelo yolo11s.pt --epocas 100 --imgsz 640 --lote 16 --dispositivo 0

## 5. Avaliar no conjunto de teste

In [ ]:
!python scripts/avaliar.py --dados dados/igarapia/data.yaml --imgsz 640

## 6. Testar em uma foto de igarapé

In [ ]:
from IPython.display import Image, display
foto = list(files.upload())[0]
!python scripts/analisar.py --fonte "$foto" --local "teste" --imgsz 960 --tta
display(Image(filename='resultados/anotada_' + os.path.splitext(foto)[0] + '.jpg'))

## 7. Baixar o modelo treinado

Coloque o arquivo baixado em `modelos/igarapia_best.pt` no seu computador. Depois disso, o painel (`streamlit run app.py`) passa a usar o modelo novo.

In [ ]:
files.download('modelos/igarapia_best.pt')
files.download('resultados/metricas_test.json')